# Structural Preprocessing

**NX-421 — Neural Signals and Signal Processing**

In this notebook, we preprocess the T1-weighted anatomical MRI from the Human Connectome Project (HCP) language-task dataset.

Structural preprocessing consists of three steps:

1. **Skull stripping:** Extract the brain by removing the skull and surrounding non-brain tissues.
2. **Tissue segmentation:** Separate the brain into gray matter (GM), white matter (WM), and cerebrospinal fluid (CSF).
3. **MNI registration:** Align the anatomical image with the MNI152 standard template.

We use FSL, as introduced in the laboratory sessions, and inspect the results after each step to check that the preprocessing worked correctly.

## 0. Setup and input data

We first import the libraries needed for image manipulation, visualization, and preprocessing.

NiBabel is used to read NIfTI images and inspect their spatial information. NumPy is used for array operations, while Matplotlib allows us to visualize the anatomical images and the preprocessing results.

For skull stripping and segmentation, we use FSL's BET and FAST tools, respectively.

The original MRI is kept unchanged, and all generated files are saved in the `outputs/structural` directory.

In [ ]:
# General imports
import os
from pathlib import Path
import subprocess
import shutil

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

# Load FSL in Neurodesk, as in Lab 2
import module as lmod

await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')

# Project directories
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "structural"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check FSL installation
print("Project directory:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

print("\nFSL tools:")

for tool in ["bet", "fast", "fslmaths"]:
    tool_path = shutil.which(tool)
    print(f"{tool}: {tool_path}")

    if tool_path is None:
        raise RuntimeError(f"{tool} is not available.")

# Check availability of FSLeyes
print("\nFSLeyes:", shutil.which("fsleyes"))

### 0.1 Loading the anatomical image

The dataset contains a T1-weighted structural MRI stored in NIfTI format (`.nii.gz`).

Before applying any preprocessing, we check the image dimensions, voxel resolution, and orientation.

This initial inspection is important because the anatomical image will be used for both brain extraction and tissue segmentation.

In [ ]:
# Path to the HCP T1-weighted anatomical image
anatomical_path = DATA_DIR / "T1" / "T1w.nii.gz"

if not anatomical_path.is_file():
    raise FileNotFoundError(
        f"Anatomical MRI not found: {anatomical_path}"
    )

# Load the anatomical MRI
t1_img = nib.load(str(anatomical_path))
t1_data = t1_img.get_fdata(dtype=np.float32)

# Verify image dimensions
if t1_data.ndim != 3:
    raise ValueError("Expected a 3D anatomical MRI.")

# Display image information
print("Anatomical image:", anatomical_path)
print("Image dimensions:", t1_img.shape)
print("Voxel size (mm):", t1_img.header.get_zooms()[:3])
print("Orientation:", nib.aff2axcodes(t1_img.affine))
print("Intensity range:", t1_data.min(), t1_data.max())

### 0.2 Initial inspection of the anatomical MRI

The T1-weighted image has dimensions of 260 × 311 × 260 voxels, with an isotropic voxel resolution of 0.7 mm.

This high spatial resolution allows us to distinguish anatomical structures and tissue boundaries in detail, which is useful for the following skull stripping and segmentation steps.

The image is oriented along the Left-Anterior-Superior (LAS) axes. Before preprocessing, we visualize the anatomical image in the three orthogonal planes to inspect the brain and surrounding tissues.

In [ ]:
# Reorient the image for consistent visualization
t1_display = nib.as_closest_canonical(t1_img).get_fdata()

# Select the central slice along each axis
x = t1_display.shape[0] // 2
y = t1_display.shape[1] // 2
z = t1_display.shape[2] // 2

# Display the three anatomical planes
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(np.rot90(t1_display[x, :, :]), cmap="gray")
axes[0].set_title("Sagittal")

axes[1].imshow(np.rot90(t1_display[:, y, :]), cmap="gray")
axes[1].set_title("Coronal")

axes[2].imshow(np.rot90(t1_display[:, :, z]), cmap="gray")
axes[2].set_title("Axial")

for ax in axes:
    ax.axis("off")

plt.suptitle("Original T1-weighted MRI")
plt.tight_layout()
plt.show()

### 0.3 Interactive Visualization with FSLeyes

As in the laboratory sessions, we also use FSLeyes to inspect the anatomical MRI interactively.

Unlike the Matplotlib figures, which show selected slices, FSLeyes allows us to navigate through the complete 3D volume and examine anatomical structures in different planes.

We first load the original T1-weighted image to inspect the brain and surrounding tissues before preprocessing.

In [ ]:
# Open FSLeyes with the original anatomical MRI
# --cliserver allows us to reuse the same window
fsleyes_process = subprocess.Popen([
    "fsleyes",
    "--cliserver",
    str(anatomical_path)
])

print("Original T1w loaded in FSLeyes.")

## 1. Skull Stripping

The first preprocessing step is skull stripping, which removes the skull and other non-brain tissues from the anatomical MRI while preserving the brain.

We use FSL's Brain Extraction Tool (BET), as introduced in Lab 2. BET estimates the brain boundary and generates a skull-stripped image together with a binary brain mask.

We enable robust brain centre estimation (`-R`) and use the default fractional intensity threshold (`-f 0.5`). The mask will then be inspected to check whether the brain has been correctly extracted.

This step is important because the following tissue segmentation will be performed on the skull-stripped image.

In [ ]:
# Output paths for skull stripping
brain_path = OUTPUT_DIR / "T1w_brain.nii.gz"
mask_path = OUTPUT_DIR / "T1w_brain_mask.nii.gz"

# Skull stripping using FSL BET
subprocess.run([
    "bet",
    str(anatomical_path),
    str(brain_path),
    "-R",
    "-m",
    "-f", "0.5"
], check=True)

# Check that the output files were created
assert brain_path.exists(), "Brain-extracted image not found."
assert mask_path.exists(), "Brain mask not found."

print("Skull stripping completed.")
print("Extracted brain:", brain_path)
print("Brain mask:", mask_path)

### 1.1 Quality Control of Skull Stripping

After applying BET, we compare the original T1-weighted image with the skull-stripped image and its corresponding brain mask.

The goal is to verify that the skull and surrounding tissues have been removed without excluding important brain regions.

We inspect the sagittal, coronal, and axial views, paying particular attention to the frontal cortex, superior regions, cerebellum, and brainstem.

The brain mask is overlaid on the original anatomical image to assess how well the extraction follows the brain boundaries.

In [ ]:
# Load the skull-stripped image and brain mask
brain_img = nib.load(str(brain_path))
mask_img = nib.load(str(mask_path))

# Check that the images are in the same space
assert brain_img.shape == t1_img.shape
assert mask_img.shape == t1_img.shape
assert np.allclose(brain_img.affine, t1_img.affine)
assert np.allclose(mask_img.affine, t1_img.affine)

# Reorient for visualization
brain_display = nib.as_closest_canonical(brain_img).get_fdata()
mask_display = nib.as_closest_canonical(mask_img).get_fdata()

# Find the approximate centre of the extracted brain
brain_center = np.rint(
    np.argwhere(mask_display > 0).mean(axis=0)
).astype(int)

fig, axes = plt.subplots(3, 3, figsize=(12, 11))

plane_names = ["Sagittal", "Coronal", "Axial"]

for axis in range(3):
    slice_index = int(brain_center[axis])

    original_slice = np.rot90(
        np.take(t1_display, slice_index, axis=axis)
    )
    brain_slice = np.rot90(
        np.take(brain_display, slice_index, axis=axis)
    )
    mask_slice = np.rot90(
        np.take(mask_display, slice_index, axis=axis)
    )

    axes[axis, 0].imshow(original_slice, cmap="gray")
    axes[axis, 1].imshow(brain_slice, cmap="gray")

    axes[axis, 2].imshow(original_slice, cmap="gray")
    axes[axis, 2].contour(
        mask_slice,
        levels=[0.5],
        colors="red",
        linewidths=0.8
    )

    for ax in axes[axis]:
        ax.axis("off")

    axes[axis, 0].set_ylabel(plane_names[axis])

axes[0, 0].set_title("Original T1w")
axes[0, 1].set_title("Skull-stripped brain")
axes[0, 2].set_title("Brain mask boundary")

plt.tight_layout()
plt.show()

### 1.2 Skull Stripping Results

The skull-stripped image was compared with the original T1-weighted MRI in sagittal, coronal, and axial views.

The BET extraction appears satisfactory in the displayed slices. The skull and surrounding tissues have been removed, while the cerebral cortex, cerebellum, and brainstem remain visible. The binary mask also follows the brain boundaries reasonably well, particularly in the frontal and superior regions.

No major extraction errors are apparent in these views, so we keep the BET parameters (`-R -f 0.5`) and use the resulting brain-extracted image for tissue segmentation.

### 1.3 Interactive Quality Control with FSLeyes

We use FSLeyes to inspect the brain extraction in more detail, following the quality-control approach used in Lab 2.

The original anatomical image is displayed together with the binary brain mask. We navigate through the different slices and check whether the mask follows the brain boundaries.

Particular attention is given to the frontal cortex, superior regions, cerebellum, and brainstem, where incorrect extraction could affect subsequent analyses.

In [ ]:
# Add the brain mask to the existing FSLeyes window
subprocess.run([
    "fsleyes",
    "--cliserver",
    str(mask_path)
], check=True)

print("Brain mask added to FSLeyes.")

## 2. Brain Tissue Segmentation

After skull stripping, we segment the brain into three main tissue classes: gray matter (GM), white matter (WM), and cerebrospinal fluid (CSF).

We use FSL FAST (FMRIB's Automated Segmentation Tool), as introduced in Lab 2. FAST uses differences in MRI intensities and spatial information to classify brain tissues.

In T1-weighted images, white matter generally appears brighter than gray matter, while CSF is darker.

FAST is applied to the skull-stripped image, since non-brain tissues could interfere with the segmentation. We request three tissue classes and inspect the resulting partial volume estimation (PVE) maps.

These maps represent the estimated proportion of each tissue within a voxel, rather than simply assigning a binary tissue label.

In [ ]:
# Output prefix for tissue segmentation
segmentation_prefix = OUTPUT_DIR / "T1w_fast"

# Run FAST on the skull-stripped T1 image
subprocess.run([
    "fast",
    "-t", "1",
    "-n", "3",
    "-o", str(segmentation_prefix),
    str(brain_path)
], check=True)

# Paths to the partial volume estimation maps
csf_path = OUTPUT_DIR / "T1w_fast_pve_0.nii.gz"
gm_path = OUTPUT_DIR / "T1w_fast_pve_1.nii.gz"
wm_path = OUTPUT_DIR / "T1w_fast_pve_2.nii.gz"

# Verify that all three maps were created
for tissue, path in [
    ("CSF", csf_path),
    ("Gray matter", gm_path),
    ("White matter", wm_path)
]:
    assert path.exists(), f"Missing output: {path}"
    print(f"{tissue}: {path.name}")

print("\nTissue segmentation completed.")

### 2.1 Interactive Visualization of Segmented Tissues

FAST generates three partial-volume estimation (PVE) maps corresponding to CSF, gray matter, and white matter.

Following Lab 2, we load these three maps in FSLeyes and overlay them on the skull-stripped anatomical image.

We use different colours to distinguish the tissues:

- CSF: Red
- Gray matter: Green
- White matter: Blue

We inspect the resulting maps to check whether the tissue classification agrees with the anatomical structures visible in the T1-weighted image.

In [ ]:
# Add the skull-stripped brain and segmented tissues
# to the same FSLeyes window
subprocess.run([
    "fsleyes",
    "--cliserver",
    str(brain_path),
    str(csf_path),
    str(gm_path),
    str(wm_path)
], check=True)

print("Segmented tissues added to FSLeyes.")

### 2.2 Visualization of Segmented Tissues

To complement the interactive inspection in FSLeyes, we also display the three tissue maps using Matplotlib.

Each partial-volume map is overlaid on the skull-stripped T1-weighted image, using red for CSF, green for gray matter, and blue for white matter.

This allows us to compare the spatial distribution of the tissues in the same anatomical slice and provides a figure for reporting the segmentation results.

In [ ]:
# Load FAST partial-volume maps
csf_img = nib.load(str(csf_path))
gm_img = nib.load(str(gm_path))
wm_img = nib.load(str(wm_path))

# Verify image geometry
for tissue_img in [csf_img, gm_img, wm_img]:
    assert tissue_img.shape == brain_img.shape
    assert np.allclose(tissue_img.affine, brain_img.affine)

# Reorient images consistently for visualization
csf_display = nib.as_closest_canonical(csf_img).get_fdata()
gm_display = nib.as_closest_canonical(gm_img).get_fdata()
wm_display = nib.as_closest_canonical(wm_img).get_fdata()

# Select an axial slice through the brain
z = int(brain_center[2])
background = np.rot90(brain_display[:, :, z])

tissues = [
    (csf_display, "CSF", "Reds"),
    (gm_display, "Gray matter", "Greens"),
    (wm_display, "White matter", "Blues")
]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

axes[0].imshow(background, cmap="gray")
axes[0].set_title("Skull-stripped T1w")

for ax, (data, title, cmap) in zip(axes[1:], tissues):
    tissue_slice = np.rot90(data[:, :, z])

    ax.imshow(background, cmap="gray")
    ax.imshow(
        np.ma.masked_less(tissue_slice, 0.2),
        cmap=cmap,
        vmin=0,
        vmax=1,
        alpha=0.7
    )

    ax.set_title(title)

for ax in axes:
    ax.axis("off")

plt.suptitle("FAST Tissue Segmentation")
plt.tight_layout()
plt.show()

### 2.3 Tissue Segmentation Results

The three partial-volume maps generated by FAST were visualized on the skull-stripped T1-weighted image.

In the inspected slices, the segmentation appears anatomically reasonable. CSF is mainly identified in the ventricles and cortical sulci, while gray matter follows the cortical ribbon and is also visible in deeper brain structures. White matter is mainly concentrated beneath the cortex and in the internal regions of the brain.

The three tissue classes are generally well separated, although partial-volume effects can be observed near tissue boundaries. No obvious major misclassification is visible in the selected slices.

Overall, the segmentation appears satisfactory based on visual inspection, although this does not guarantee accurate classification throughout the entire brain.

### 2.4 Quantitative Quality Control

In addition to visual inspection, we check the range of the partial-volume estimates and calculate approximate tissue volumes.

Each tissue fraction should lie between 0 and 1. Within the extracted brain, the fractions of CSF, GM, and WM should approximately sum to 1.

The tissue volumes are estimated by summing the partial-volume fractions and multiplying by the physical volume of one voxel.

In [ ]:
# Tissue maps as NumPy arrays
tissue_maps = {
    "CSF": csf_img.get_fdata(dtype=np.float32),
    "Gray matter": gm_img.get_fdata(dtype=np.float32),
    "White matter": wm_img.get_fdata(dtype=np.float32)
}

# Volume of a single voxel in mm³
voxel_volume = np.prod(t1_img.header.get_zooms()[:3])

print("FAST segmentation summary")
print("-" * 45)

for tissue, data in tissue_maps.items():

    if not np.isfinite(data).all():
        raise ValueError(f"Invalid values found in {tissue} map.")

    if data.min() < -1e-4 or data.max() > 1.0001:
        raise ValueError(f"Unexpected PVE values in {tissue} map.")

    # Estimated tissue volume from partial-volume fractions
    volume_ml = np.sum(data) * voxel_volume / 1000

    print(f"{tissue}:")
    print(f"  PVE range: {data.min():.3f} - {data.max():.3f}")
    print(f"  Estimated volume: {volume_ml:.1f} mL")

# Check whether tissue fractions approximately sum to 1 inside the brain
brain_mask = mask_img.get_fdata() > 0

pve_sum = sum(tissue_maps.values())

print("\nMean tissue fraction sum inside brain:",
      np.mean(pve_sum[brain_mask]))

### 2.5 Quantitative Results

The estimated tissue volumes were 203.7 mL for CSF, 627.4 mL for gray matter, and 490.5 mL for white matter.

All three PVE maps contained values between 0 and 1, as expected. The mean sum of tissue fractions inside the brain mask was 1.0, which is consistent with FAST's three-class partial-volume estimation.

The estimated tissue volumes appear plausible, and the results are consistent with the anatomical patterns observed during visual inspection. However, these checks do not provide an independent measure of segmentation accuracy, particularly at tissue boundaries.